# ATML PA2 - Task 2: PPO

**Before running:** Settings → Accelerator **GPU T4** (x1 or x2), Internet **On**.

Rough T4 time: ~1.5-2.5 h. Run cells top to bottom; the last cell zips everything to download.
`outputs/task2_ppo/standard` is the PPO policy needed later in Task 4 - keep the zip.

## 1. Setup

In [ ]:
# Fresh session: clone; re-run: pull the latest code.
%cd /kaggle/working
!if [ -d pa2 ]; then git -C pa2 pull; else git clone -b pa2 https://github.com/rana-taqveem/atml_internal.git pa2; fi
%cd /kaggle/working/pa2
!pip install -q -r requirements.txt
!python -m scripts.download_assets
!python -m scripts.validate_assets

In [ ]:
!python -m scripts.check_environment   # expect CUDA available: True, GPU: Tesla T4

## 2. Smoke test (~5 min)
Checks GPU memory (policy + critic + reward model) and the 768-token evaluation.

In [ ]:
!python -m task2_ppo.continue_train --config configs/ppo.yaml --run-name smoke --output outputs/task2_ppo/smoke --updates 2
!python -m task2_ppo.evaluate --config configs/ppo.yaml --adapter outputs/task2_ppo/smoke --name smoke --n-prompts 8
!head -40 results/task2_ppo/smoke/train_summary.json

In [ ]:
!rm -rf outputs/task2_ppo/smoke outputs/task2_ppo/smoke_value results/task2_ppo/smoke

## 3. Step 1 - standard 20-update continuation + held-out evaluation
Evaluates the supplied midpoint, the untouched SFT model and the continued policy on the same prompts.

In [ ]:
!python -m task2_ppo.continue_train --config configs/ppo.yaml --run-name standard 2>&1 | tee -a task2.log
!python -m task2_ppo.evaluate --config configs/ppo.yaml --adapter checkpoints/ppo_midpoint_policy --name midpoint 2>&1 | tee -a task2.log
!python -m task2_ppo.evaluate --config configs/ppo.yaml --adapter none --name sft 2>&1 | tee -a task2.log
!python -m task2_ppo.evaluate --config configs/ppo.yaml --adapter outputs/task2_ppo/standard --name standard 2>&1 | tee -a task2.log

## 4. Step 2 - clipping study (cached batch + eps forks)

In [ ]:
!python -m task2_ppo.analyze_clipping --config configs/ppo.yaml --skip-existing 2>&1 | tee -a task2.log

## 5. Step 3 - KL-pressure forks (the kl_beta=0.10 fork is reused from step 2)

In [ ]:
!python -m task2_ppo.ablate_kl --config configs/ppo.yaml --skip-existing 2>&1 | tee -a task2.log

## 6. Summary, figures, qualitative candidates

In [ ]:
!python -m task2_ppo.summarize --config configs/ppo.yaml 2>&1 | tee -a task2.log

## 7. Save (download `task2_outputs.zip` from the Output panel)

In [ ]:
!zip -qr /kaggle/working/task2_outputs.zip outputs/task2_ppo results/task2_ppo task2.log
!ls -lh /kaggle/working/task2_outputs.zip